# HDFS Explanation Pipeline

Complete Screener-Reasoner pipeline for HDFS dataset with:
- AllLinLog screener for anomaly detection
- BM25 evidence retrieval (RAG)
- LLM-based explanation generation
- Evidence-grounded verification

Based on `03_pipeline_complete.ipynb` (BGL version).

## 1. Imports

In [21]:
# Standard library
import sys
import json
import time
from pathlib import Path
from importlib import reload
from collections import Counter

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

# Force reload modules to pick up fixes
from src import screener as screener_module
from src import prompt_builder as prompt_builder_module
reload(screener_module)
reload(prompt_builder_module)

# Project imports
from src.data_loader import HDFSDataLoader, Session
from src.screener import Screener, ScreenerOutput
from src.evidence_store import EvidenceStore
from src.retriever import BM25Retriever
from src.prompt_builder import PromptBuilder, TraceExplanation, Claim, Signature
from src.llm_client import LLMClient
from src.verifier import Verifier

from tqdm import tqdm

print("All imports successful")

All imports successful


## 2. Data Loading

In [7]:
# Load HDFS dataset
loader = HDFSDataLoader(
    log_file='../logs/HDFS.log',
    label_file='../logs/anomaly_label_HDFS.csv'
)
loader.load()

# Get splits
train_sessions = loader.get_train()
test_sessions = loader.get_test()

# Statistics
train_anomaly = sum(1 for s in train_sessions if s.label == 1)
test_anomaly = sum(1 for s in test_sessions if s.label == 1)

print(f"Train: {len(train_sessions):,} sessions ({train_anomaly:,} anomalies, {train_anomaly/len(train_sessions):.2%})")
print(f"Test:  {len(test_sessions):,} sessions ({test_anomaly:,} anomalies, {test_anomaly/len(test_sessions):.2%})")

Loading HDFS logs from: ../logs/HDFS.log
Loading labels from: ../logs/anomaly_label_HDFS.csv


Reading HDFS logs: 11175629it [00:10, 1097589.04it/s]


Found 575061 unique blocks
Train: 402,542 sessions (11,786 anomalies, 2.93%)
Test:  86,260 sessions (2,526 anomalies, 2.93%)


## 3. Screener

In [8]:
# Load pre-trained screener model
screener = Screener.from_pretrained(
    dataset="HDFS",
    model_path="../best_model_HDFS/best_model_HDFS20250804_201746.pth"
)

total_params = sum(p.numel() for p in screener.model.parameters())
print(f"Model parameters: {total_params:,}")

Loading Screener for HDFS on cuda
Loading cl100k_base (GPT-4) tokenizer...
Loading model weights from: ../best_model_HDFS/best_model_HDFS20250804_201746.pth
Model loaded! Parameters: 15,501,506
Model parameters: 15,501,506


In [9]:
# Screen test sessions
sample_size = 500
sample_sessions = test_sessions[:sample_size]

print(f"Screening {sample_size} sessions...")
start = time.time()
screener_outputs = screener.screen_sessions(sample_sessions)
elapsed = time.time() - start

print(f"Done in {elapsed:.2f}s ({elapsed/sample_size*1000:.1f}ms per session)")

# Count predictions
predicted_anomalies = [o for o in screener_outputs if o.is_anomaly]
print(f"\nPredicted anomalies: {len(predicted_anomalies)} / {sample_size}")

# Ground truth in sample
gt_anomalies = sum(1 for s in sample_sessions if s.label == 1)
print(f"Ground truth anomalies: {gt_anomalies} / {sample_size}")

# Quick accuracy check
correct = sum(1 for s, o in zip(sample_sessions, screener_outputs) if s.label == o.pred)
print(f"Accuracy: {correct/sample_size:.2%}")

Screening 500 sessions...


Screening sessions: 100%|██████████| 63/63 [00:00<00:00, 73.08it/s]

Done in 0.86s (1.7ms per session)

Predicted anomalies: 17 / 500
Ground truth anomalies: 17 / 500
Accuracy: 100.00%


## 4. Evidence Store

In [10]:
# Build evidence store from training sessions
evidence_store = EvidenceStore(dataset="HDFS")
evidence_store.build_from_sessions(train_sessions, show_progress=True)

# Show statistics
stats = evidence_store.stats()
print("\nEvidence Store Stats:")
for key, value in stats.items():
    print(f"  {key}: {value:,}" if isinstance(value, int) else f"  {key}: {value}")

Building evidence store: 100%|██████████| 402542/402542 [02:38<00:00, 2532.24it/s]


Evidence store built with 402542 documents

Evidence Store Stats:
  total_documents: 402,542
  normal_documents: 390,756
  anomaly_documents: 11,786
  by_evidence_type: {'session': 402542, 'signature': 0, 'profile': 0}
  avg_text_length: 2066.1526250676948
  min_text_length: 208
  max_text_length: 29,077


## 5. Retriever (RAG)

In [11]:
# Build BM25 retriever
retriever = BM25Retriever(evidence_store)
retriever.build_index()
print(f"BM25 index built with {len(evidence_store.documents):,} documents")

Building BM25 index...
BM25 index built with 402542 documents
BM25 index built with 402,542 documents


In [12]:
# Test retrieval on first predicted anomaly
if predicted_anomalies:
    test_idx = screener_outputs.index(predicted_anomalies[0])
    test_session = sample_sessions[test_idx]
    scr_output = predicted_anomalies[0]
    
    print(f"Test session: {test_session.session_id}")
    print(f"Session lines: {len(test_session.lines)}")
    print(f"Ground truth: {'ANOMALY' if test_session.label == 1 else 'NORMAL'}")
    
    # Mixed retrieval (4 anomaly + 1 normal)
    print("\n=== Mixed Retrieval (4 anomaly + 1 normal) ===")
    mixed_hits = retriever.retrieve_for_session_mixed(test_session, top_k_anomaly=4, top_k_normal=1)
    for h in mixed_hits:
        label = "anomaly" if h.metadata.get("label") == 1 else "normal"
        print(f"  {h.evidence_id[:25]:25s} label={label:7s} score={h.score:.2f}")
else:
    print("No predicted anomalies found - check screener")

Test session: HDFS_blk_-9153926305989047396
Session lines: 27
Ground truth: ANOMALY

=== Mixed Retrieval (4 anomaly + 1 normal) ===
  E_HDFS_blk_-7158945724117 label=anomaly score=993.82
  E_HDFS_blk_-2015623546668 label=anomaly score=985.51
  E_HDFS_blk_-5138448476301 label=anomaly score=985.51
  E_HDFS_blk_79773830736486 label=anomaly score=985.51
  E_HDFS_blk_21837106398303 label=normal  score=976.82


## 6. Prompt Builder

In [ ]:
# Initialize prompt builder with HDFS-specific prompts
builder = PromptBuilder(
    max_log_lines=20,
    max_chars_per_evidence=500,
    max_evidence_items=5,
    dataset="HDFS"  # Use HDFS-specific signature examples
)

# Build prompt for test session
if predicted_anomalies:
    system_prompt, user_prompt = builder.build_prompt(
        session=test_session,
        screener_output=scr_output,
        evidence_hits=mixed_hits
    )
    
    print("=== SYSTEM PROMPT ===")
    print(system_prompt[:800])
    print("...")
    
    print("\n=== USER PROMPT (truncated) ===")
    print(user_prompt[:1500])
    print("...")

=== SYSTEM PROMPT ===
You are an expert log analyst producing forensic, evidence-grounded explanations.
Your task is to explain WHY a log session is anomalous based on the provided evidence.

EVIDENCE FORMAT:
- Each evidence block has LINE NUMBERS: E0-L1, E0-L2, E1-L1, E1-L2, etc.
- [E0] = The query session being analyzed
- [E1], [E2], ... = Retrieved historical evidence (may include anomaly or normal sessions)

CLAIM TYPES (you MUST produce at least one of each type when evidence allows):
- "observation": Direct obs
...

=== USER PROMPT (truncated) ===
Analyze this LOG SESSION that was flagged as ANOMALOUS by our detection model.

=== [E0] QUERY SESSION TO ANALYZE ===
Session ID: HDFS_blk_-9153926305989047396
Anomaly Probability: 100.00%
Confidence Margin: 1.0000

Log Content (with line numbers):
E0-L1: 081110 210913 11399 INFO dfs.DataNode$DataXceiver: Receiving block blk_-9153926305989047396 src: /10.251.126.255:54606 dest: /10.251.126.255:50010
E0-L2: 081110 210913 14236 INFO dfs.D

## 7. LLM Client

In [14]:
# Initialize LLM client
llm_client = LLMClient(
    provider="ollama",
    model="llama3.1:8b",
    temperature=0.1,
    max_tokens=1024,
    timeout=120
)

# Check availability
if llm_client.is_available():
    print(f"LLM ({llm_client.model}) is available")
else:
    print(f"LLM not available. Start with: ollama serve")

LLM (llama3.1:8b) is available


In [15]:
# Generate explanation for test session
if predicted_anomalies and llm_client.is_available():
    print("Generating explanation...")
    start = time.time()
    
    response = llm_client.generate(
        prompt=user_prompt,
        system_prompt=system_prompt,
        json_mode=True
    )
    
    elapsed = time.time() - start
    print(f"Done in {elapsed:.2f}s")
    print(f"Tokens: {response.total_tokens}")

Generating explanation...
Done in 15.24s
Tokens: 3497


In [16]:
# Parse and display the explanation
if predicted_anomalies and llm_client.is_available():
    explanation_dict = json.loads(response.content)
    
    print("=" * 60)
    print("LLM EXPLANATION")
    print("=" * 60)
    
    # Signature
    if 'signature' in explanation_dict:
        sig = explanation_dict['signature']
        print(f"\nSignature: {sig.get('name', 'N/A')}")
    
    print(f"\nPrediction: {explanation_dict.get('prediction')}")
    print(f"Summary: {explanation_dict.get('summary')}")
    print(f"\nClaims ({len(explanation_dict.get('claims', []))}):")
    
    for i, claim in enumerate(explanation_dict.get('claims', []), 1):
        print(f"\n  [{i}] {claim.get('type', 'observation')}")
        print(f"      {claim.get('claim', 'N/A')}")
        print(f"      Evidence: {claim.get('evidence_ids', [])} | Spans: {claim.get('evidence_spans', [])}")

LLM EXPLANATION

Signature: RAS_KERNEL_FATAL__DATA_STORAGE_INTERRUPT

Prediction: anomaly
Summary: RAS_KERNEL_FATAL__DATA_STORAGE_INTERRUPT: 3 FATAL errors in E0-L8 to E0-L12, matching signature from E1.

Claims (3):

  [1] observation
      E0 contains 3 KERNEL FATAL errors with 'data storage interrupt' concentrated in lines 8-12.
      Evidence: ['E0'] | Spans: ['E0-L10', 'E0-L11', 'E0-L12']

  [2] pattern_match
      The combination {KERNEL FATAL + data storage interrupt + instruction address} matches historical anomaly signature RAS_KERNEL_FATAL__DATA_STORAGE_INTERRUPT.
      Evidence: ['E0', 'E1'] | Spans: ['E0-L10', 'E0-L11', 'E1-L3']

  [3] contrast
      E0 has 'FATAL + interrupt' at E0-L8; normal E5 shows 'INFO + corrected' at E5-L4 without escalation.
      Evidence: ['E0', 'E5'] | Spans: ['E0-L8', 'E5-L3']


## 8. Verifier

In [17]:
# Initialize verifier and verify explanation
verifier = Verifier()

if predicted_anomalies and llm_client.is_available():
    # Convert dict to TraceExplanation
    sig_dict = explanation_dict.get('signature')
    signature = None
    if sig_dict:
        signature = Signature(
            name=sig_dict.get('name', 'UNKNOWN'),
            matched_evidence_ids=sig_dict.get('matched_evidence_ids', [])
        )
    
    trace_exp = TraceExplanation(
        prediction=explanation_dict.get('prediction'),
        summary=explanation_dict.get('summary'),
        signature=signature,
        claims=[Claim(
            type=c.get('type', 'observation'),
            claim=c.get('claim'),
            evidence_ids=c.get('evidence_ids', []),
            evidence_spans=c.get('evidence_spans', [])
        ) for c in explanation_dict.get('claims', [])],
        insufficient_evidence=explanation_dict.get('insufficient_evidence', False)
    )
    
    # Build evidence ID mapping and verify
    evidence_id_mapping = builder.build_evidence_id_mapping(test_session, mixed_hits)
    query_session_text = "\n".join(test_session.lines)
    
    verification = verifier.verify(
        explanation=trace_exp,
        evidence_hits=mixed_hits,
        evidence_id_mapping=evidence_id_mapping,
        query_session_text=query_session_text
    )
    
    print("=" * 60)
    print("VERIFICATION RESULT")
    print("=" * 60)
    print(f"\nPassed: {verification.passed}")
    print(f"Checks: {verification.passed_checks}/{verification.total_checks} passed")

VERIFICATION RESULT

Passed: True
Checks: 8/8 passed


## 9. Complete Pipeline Function

In [18]:
def explain_session(
    session: Session,
    screener_output: ScreenerOutput,
    retriever: BM25Retriever,
    builder: PromptBuilder,
    llm_client: LLMClient,
    verifier: Verifier
) -> dict:
    """
    Generate and verify an explanation for an anomalous session.
    
    Pipeline steps:
    1. Retrieve evidence (4 anomaly + 1 normal) via BM25
    2. Build structured prompt with evidence
    3. Call LLM to generate explanation
    4. Parse and validate response
    5. Verify faithfulness against evidence
    
    Returns:
        dict with explanation, verification status, and metrics
    """
    start = time.time()
    
    # 1. Retrieve evidence (mixed: 4 anomaly + 1 normal)
    evidence_hits = retriever.retrieve_for_session_mixed(
        session, 
        top_k_anomaly=4, 
        top_k_normal=1
    )
    
    # 2. Build prompt
    system_prompt, user_prompt = builder.build_prompt(
        session=session,
        screener_output=screener_output,
        evidence_hits=evidence_hits
    )
    
    # 3. Call LLM
    response = llm_client.generate(
        prompt=user_prompt,
        system_prompt=system_prompt,
        json_mode=True
    )
    
    # 4. Parse response
    try:
        explanation_dict = json.loads(response.content)
        parse_success = True
    except json.JSONDecodeError:
        explanation_dict = {"prediction": "anomaly", "summary": "Parse error", "claims": [], "signature": None}
        parse_success = False
    
    # 5. Convert to TraceExplanation
    sig_dict = explanation_dict.get('signature')
    signature = Signature(
        name=sig_dict.get('name', 'UNKNOWN'),
        matched_evidence_ids=sig_dict.get('matched_evidence_ids', [])
    ) if sig_dict else None
    
    trace_exp = TraceExplanation(
        prediction=explanation_dict.get('prediction'),
        summary=explanation_dict.get('summary'),
        signature=signature,
        claims=[Claim(
            type=c.get('type', 'observation'),
            claim=c.get('claim'),
            evidence_ids=c.get('evidence_ids', []),
            evidence_spans=c.get('evidence_spans', [])
        ) for c in explanation_dict.get('claims', [])],
        insufficient_evidence=explanation_dict.get('insufficient_evidence', False)
    )
    
    # 6. Verify
    evidence_id_mapping = builder.build_evidence_id_mapping(session, evidence_hits)
    query_session_text = "\n".join(session.lines)
    verification = verifier.verify(
        trace_exp, evidence_hits, evidence_id_mapping,
        query_session_text=query_session_text
    )
    
    elapsed = time.time() - start
    
    return {
        'session_id': session.session_id,
        'explanation': explanation_dict,
        'signature': signature.name if signature else None,
        'verification_passed': verification.passed,
        'verification_details': {
            'total_checks': verification.total_checks,
            'passed_checks': verification.passed_checks,
            'failed_checks': verification.failed_checks,
            'warning_checks': verification.warning_checks
        },
        'parse_success': parse_success,
        'tokens': response.total_tokens,
        'latency_ms': elapsed * 1000
    }

print("Pipeline function defined: explain_session()")

Pipeline function defined: explain_session()


In [19]:
# Test the complete pipeline function
if predicted_anomalies and llm_client.is_available():
    result = explain_session(
        session=test_session,
        screener_output=scr_output,
        retriever=retriever,
        builder=builder,
        llm_client=llm_client,
        verifier=verifier
    )
    
    print("=" * 60)
    print("PIPELINE RESULT")
    print("=" * 60)
    print(f"Session: {result['session_id']}")
    print(f"Signature: {result['signature']}")
    print(f"Parse success: {result['parse_success']}")
    print(f"Verification: {'PASSED' if result['verification_passed'] else 'FAILED'}")
    print(f"Tokens: {result['tokens']}")
    print(f"Latency: {result['latency_ms']:.0f}ms")

PIPELINE RESULT
Session: HDFS_blk_-9153926305989047396
Signature: RAS_KERNEL_FATAL__DATA_STORAGE_INTERRUPT
Parse success: True
Verification: PASSED
Tokens: 3509
Latency: 36135ms


## 10. Batch Processing

In [20]:
# Batch process predicted anomalies
if predicted_anomalies and llm_client.is_available():
    # Get all predicted anomalies
    anomaly_pairs = [
        (sample_sessions[i], screener_outputs[i])
        for i, o in enumerate(screener_outputs)
        if o.is_anomaly
    ]
    
    # Limit batch size for demo
    max_batch = min(20, len(anomaly_pairs))
    anomaly_pairs = anomaly_pairs[:max_batch]
    
    print(f"Processing {len(anomaly_pairs)} anomalies...")
    
    batch_results = []
    for session, scr_out in tqdm(anomaly_pairs, desc="Explaining"):
        result = explain_session(
            session=session,
            screener_output=scr_out,
            retriever=retriever,
            builder=builder,
            llm_client=llm_client,
            verifier=verifier
        )
        batch_results.append(result)
    
    # Summary statistics
    passed = sum(1 for r in batch_results if r['verification_passed'])
    total_tokens = sum(r['tokens'] for r in batch_results)
    avg_latency = sum(r['latency_ms'] for r in batch_results) / len(batch_results)
    
    print("\n" + "=" * 60)
    print("BATCH RESULTS")
    print("=" * 60)
    print(f"\nSessions processed: {len(batch_results)}")
    print(f"Verification passed: {passed} / {len(batch_results)} ({passed/len(batch_results):.1%})")
    print(f"Total tokens: {total_tokens:,}")
    print(f"Avg tokens/session: {total_tokens/len(batch_results):.0f}")
    print(f"Avg latency: {avg_latency:.0f}ms")
    
    # Signature distribution
    signatures = [r['signature'] for r in batch_results if r['signature']]
    sig_counts = Counter(signatures)
    print(f"\nSignature distribution:")
    for sig, count in sig_counts.most_common(10):
        print(f"  {sig}: {count}")
    
    # Verification breakdown
    total_checks = sum(r['verification_details']['total_checks'] for r in batch_results)
    passed_checks = sum(r['verification_details']['passed_checks'] for r in batch_results)
    print(f"\nVerification checks: {passed_checks}/{total_checks} ({passed_checks/total_checks:.1%})")
else:
    print("No anomalies to process or LLM unavailable")

Processing 17 anomalies...


Explaining: 100%|██████████| 17/17 [06:54<00:00, 24.40s/it]


BATCH RESULTS

Sessions processed: 17
Verification passed: 13 / 17 (76.5%)
Total tokens: 52,130
Avg tokens/session: 3066
Avg latency: 24400ms

Signature distribution:
  RAS_KERNEL_FATAL__DATA_STORAGE_INTERRUPT: 17

Verification checks: 128/136 (94.1%)


## 11. Summary

This notebook demonstrates the complete HDFS explanation pipeline:

1. **Data Loading**: HDFS log sessions with block-based grouping
2. **Screener**: AllLinLog model (99.95% accuracy on test set)
3. **Evidence Store**: BM25-indexed training sessions
4. **Retrieval**: Mixed anomaly/normal evidence for contrast
5. **LLM Explanation**: Structured JSON with claims and signatures
6. **Verification**: Evidence-grounded faithfulness checks

Key metrics to track:
- Screener accuracy/recall
- Verification pass rate
- Signature distribution
- Latency and token usage

# HDFS Explanation Pipeline

Complete Screener-Reasoner pipeline for HDFS dataset.
Based on `03_pipeline_complete.ipynb` (BGL version).

## 1. Imports

In [1]:
# Standard library
import sys
import json
import time
from pathlib import Path
import importlib

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

# Force reload screener module to pick up segment clamping fix
import src.screener
importlib.reload(src.screener)

# Project imports
from src.data_loader import HDFSDataLoader, Session
from src.screener import Screener, ScreenerOutput
from src.evidence_store import EvidenceStore
from src.retriever import BM25Retriever, Retriever
from src.prompt_builder import PromptBuilder, TraceExplanation, Claim, Signature, format_evidence_block
from src.llm_client import LLMClient
from src.verifier import Verifier

print("All imports successful")

All imports successful


## 2. Data Loading

In [2]:
# Load HDFS dataset
loader = HDFSDataLoader(
    log_file='../logs/HDFS.log',
    label_file='../logs/anomaly_label_HDFS.csv'
)
loader.load()

# Get splits
train_sessions = loader.get_train()
test_sessions = loader.get_test()

# Statistics
train_anomaly = sum(1 for s in train_sessions if s.label == 1)
test_anomaly = sum(1 for s in test_sessions if s.label == 1)

print(f"Train: {len(train_sessions):,} sessions ({train_anomaly:,} anomalies, {train_anomaly/len(train_sessions):.2%})")
print(f"Test:  {len(test_sessions):,} sessions ({test_anomaly:,} anomalies, {test_anomaly/len(test_sessions):.2%})")

Loading HDFS logs from: ../logs/HDFS.log
Loading labels from: ../logs/anomaly_label_HDFS.csv


Reading HDFS logs: 11175629it [00:12, 911866.31it/s] 


Found 575061 unique blocks
Train: 402,542 sessions (11,786 anomalies, 2.93%)
Test:  86,260 sessions (2,526 anomalies, 2.93%)


## 3. Screener

In [3]:
# Load pre-trained screener model
screener = Screener.from_pretrained(
    dataset="HDFS",
    model_path="../best_model_HDFS/best_model_HDFS20250804_201746.pth"
)

total_params = sum(p.numel() for p in screener.model.parameters())
print(f"Model parameters: {total_params:,}")

Loading Screener for HDFS on cuda
Loading cl100k_base (GPT-4) tokenizer...
Loading model weights from: ../best_model_HDFS/best_model_HDFS20250804_201746.pth
Model loaded! Parameters: 15,501,506
Model parameters: 15,501,506


In [4]:
# Screen a sample of test sessions
sample_size = 200
sample_sessions = test_sessions[:sample_size]

print(f"Screening {sample_size} sessions...")
start = time.time()
screener_outputs = screener.screen_sessions(sample_sessions)
elapsed = time.time() - start

print(f"Done in {elapsed:.2f}s ({elapsed/sample_size*1000:.1f}ms per session)")

# Count predictions
predicted_anomalies = [o for o in screener_outputs if o.is_anomaly]
print(f"\nPredicted anomalies: {len(predicted_anomalies)} / {sample_size}")

# Ground truth
gt_anomalies = sum(1 for s in sample_sessions if s.label == 1)
print(f"Ground truth anomalies: {gt_anomalies} / {sample_size}")

Screening 200 sessions...


Screening sessions: 100%|██████████| 25/25 [00:00<00:00, 32.12it/s]

Done in 0.78s (3.9ms per session)

Predicted anomalies: 11 / 200
Ground truth anomalies: 11 / 200


In [5]:
# Debug: Use EXACT same processing as HDFS_screener.ipynb 
import torch
from datetime import datetime
import re
from sklearn.model_selection import train_test_split

# Re-process data exactly like HDFS_screener.ipynb
LOG_FILE = "../logs/HDFS.log"
LABEL_FILE = "../logs/anomaly_label_HDFS.csv"
MAX_TOKEN_LENGTH = 18000

print("Processing data exactly like HDFS_screener.ipynb...")

# Load labels
import pandas as pd
label_df = pd.read_csv(LABEL_FILE, engine='c', na_filter=False)
label_df = label_df.set_index("BlockId")
label_mapping = label_df["Label"].to_dict()

# Read logs and group by block_id with timestamp
session_dict = {}
with open(LOG_FILE, mode="r", encoding='utf8') as f:
    for line in tqdm(f, desc="Reading logs"):
        line = line.strip()
        tokens = line.split()
        if len(tokens) < 2:
            continue
        try:
            timestamp_str = " ".join(tokens[:2])
            timestamp = datetime.strptime(timestamp_str, '%y%m%d %H%M%S').timestamp()
        except:
            continue
        
        blk_ids = list(set(re.findall(r'(blk_-?\d+)', line)))
        if len(blk_ids) != 1:
            continue
        blk_id = blk_ids[0]
        
        if blk_id not in session_dict:
            session_dict[blk_id] = []
        session_dict[blk_id].append((timestamp, line))

print(f"Found {len(session_dict)} blocks")

# Process sessions with same tokenization
tokenizer = screener.tokenizer
allowed_special = {"<|startoftext|>", "<|endoftext|>"}
bos_token = tokenizer.encode("<|startoftext|>", allowed_special=allowed_special)[0]
eos_token = tokenizer.encode("<|endoftext|>", allowed_special=allowed_special)[0]

def tokenize_session(log_sequence, max_len=18000):
    input_ids = []
    segment_ids = []
    for i, log in enumerate(log_sequence):
        tokens = tokenizer.encode(log, allowed_special=allowed_special)
        if i == 0:
            tokens = [bos_token] + tokens
        tokens = tokens + [eos_token]
        input_ids.extend(tokens)
        segment_ids.extend([i] * len(tokens))
    if len(input_ids) > max_len:
        input_ids = input_ids[:max_len]
        segment_ids = segment_ids[:max_len]
    return input_ids, segment_ids

sessions = []
for blk_id, events in tqdm(session_dict.items(), desc="Processing sessions"):
    events.sort(key=lambda x: x[0])
    log_sequence = [msg for (ts, msg) in events]
    session_label = 1 if label_mapping.get(blk_id, "Normal") == "Anomaly" else 0
    input_ids, segment_ids = tokenize_session(log_sequence, MAX_TOKEN_LENGTH)
    sessions.append({
        "block_id": blk_id,
        "input_ids": input_ids,
        "segment_ids": segment_ids,
        "session_label": session_label
    })

print(f"Total sessions: {len(sessions)}")

# Use EXACT same train_test_split as HDFS_screener
session_labels = [s["session_label"] for s in sessions]
train_sessions_raw, temp_sessions, train_labels, temp_labels = train_test_split(
    sessions, session_labels,
    test_size=0.3,
    stratify=session_labels,
    random_state=42
)
val_relative = 0.15 / 0.30
temp_labels = [s["session_label"] for s in temp_sessions]
val_sessions_raw, test_sessions_raw, _, _ = train_test_split(
    temp_sessions, temp_labels,
    test_size=(1 - val_relative),
    stratify=temp_labels,
    random_state=42
)

print(f"Test sessions: {len(test_sessions_raw)}")
test_anomalies_raw = [s for s in test_sessions_raw if s["session_label"] == 1]
print(f"Test anomalies: {len(test_anomalies_raw)}")

Processing data exactly like HDFS_screener.ipynb...


NameError: name 'tqdm' is not defined

In [31]:
# Test multiple anomalies
import torch.nn.functional as F

MAX_SEQ_LEN = 15166
MAX_SEGMENT = 298

print("Testing first 10 anomalies from exact test split:")
print("="*60)

for idx in range(10):
    test_anomaly = test_anomalies_raw[idx]
    
    # Prepare input
    input_ids = torch.tensor([test_anomaly['input_ids']], dtype=torch.long)
    segment_ids = torch.tensor([test_anomaly['segment_ids']], dtype=torch.long)
    
    if input_ids.size(1) < MAX_SEQ_LEN:
        pad_len = MAX_SEQ_LEN - input_ids.size(1)
        input_ids = F.pad(input_ids, (0, pad_len), value=0)
        segment_ids = F.pad(segment_ids, (0, pad_len), value=0)
    else:
        input_ids = input_ids[:, :MAX_SEQ_LEN]
        segment_ids = segment_ids[:, :MAX_SEQ_LEN]
    
    segment_ids = torch.clamp(segment_ids, 0, MAX_SEGMENT - 1)
    
    input_ids = input_ids.to(screener.device)
    segment_ids = segment_ids.to(screener.device)
    attention_mask = (input_ids != 0).long()
    
    with torch.no_grad():
        logits = screener.model(input_ids, segment_ids, position_ids=None, attention_mask=attention_mask)
        probs = torch.softmax(logits, dim=1)
    
    pred = "ANOMALY" if logits.argmax(dim=1).item() == 1 else "NORMAL"
    prob_anomaly = probs[0, 1].item()
    print(f"{idx+1}. {test_anomaly['block_id'][:30]:30s} tokens={len(test_anomaly['input_ids']):5d} → {pred:7s} (P(anom)={prob_anomaly:.2e})")

# Also test some normal sessions
print("\n" + "="*60)
print("Testing first 5 NORMAL sessions from test split:")
print("="*60)
test_normals = [s for s in test_sessions_raw if s["session_label"] == 0][:5]

for idx, test_normal in enumerate(test_normals):
    input_ids = torch.tensor([test_normal['input_ids']], dtype=torch.long)
    segment_ids = torch.tensor([test_normal['segment_ids']], dtype=torch.long)
    
    if input_ids.size(1) < MAX_SEQ_LEN:
        pad_len = MAX_SEQ_LEN - input_ids.size(1)
        input_ids = F.pad(input_ids, (0, pad_len), value=0)
        segment_ids = F.pad(segment_ids, (0, pad_len), value=0)
    else:
        input_ids = input_ids[:, :MAX_SEQ_LEN]
        segment_ids = segment_ids[:, :MAX_SEQ_LEN]
    
    segment_ids = torch.clamp(segment_ids, 0, MAX_SEGMENT - 1)
    
    input_ids = input_ids.to(screener.device)
    segment_ids = segment_ids.to(screener.device)
    attention_mask = (input_ids != 0).long()
    
    with torch.no_grad():
        logits = screener.model(input_ids, segment_ids, position_ids=None, attention_mask=attention_mask)
        probs = torch.softmax(logits, dim=1)
    
    pred = "ANOMALY" if logits.argmax(dim=1).item() == 1 else "NORMAL"
    prob_anomaly = probs[0, 1].item()
    print(f"{idx+1}. {test_normal['block_id'][:30]:30s} tokens={len(test_normal['input_ids']):5d} → {pred:7s} (P(anom)={prob_anomaly:.2e})")

Block: blk_6279226377833208445
Label: 1
Tokens: 941
Padded shape: torch.Size([1, 15166])

Logits: tensor([[ 8.7922, -8.1813]], device='cuda:0')
Probs: tensor([[1.0000e+00, 4.2511e-08]], device='cuda:0')
Prediction: NORMAL


# Test inference on exact test set anomaly
import torch.nn.functional as F

MAX_SEQ_LEN = 15166  # From checkpoint
MAX_SEGMENT = 298

# Get first test anomaly
test_anomaly = test_anomalies_raw[0]
print(f"Block: {test_anomaly['block_id']}")
print(f"Label: {test_anomaly['session_label']}")
print(f"Tokens: {len(test_anomaly['input_ids'])}")

# Prepare input exactly like HDFS_screener collate_fn
input_ids = torch.tensor([test_anomaly['input_ids']], dtype=torch.long)
segment_ids = torch.tensor([test_anomaly['segment_ids']], dtype=torch.long)

# Pad/truncate to MAX_SEQ_LEN
if input_ids.size(1) < MAX_SEQ_LEN:
    pad_len = MAX_SEQ_LEN - input_ids.size(1)
    input_ids = F.pad(input_ids, (0, pad_len), value=0)
    segment_ids = F.pad(segment_ids, (0, pad_len), value=0)
else:
    input_ids = input_ids[:, :MAX_SEQ_LEN]
    segment_ids = segment_ids[:, :MAX_SEQ_LEN]

# Clamp segment_ids
segment_ids = torch.clamp(segment_ids, 0, MAX_SEGMENT - 1)

print(f"Padded shape: {input_ids.shape}")

# Move to device
input_ids = input_ids.to(screener.device)
segment_ids = segment_ids.to(screener.device)
attention_mask = (input_ids != 0).long()

# Create position_ids
position_ids = torch.arange(MAX_SEQ_LEN, device=screener.device).unsqueeze(0)

# Inference
with torch.no_grad():
    logits = screener.model(input_ids, segment_ids, position_ids, attention_mask)
    probs = torch.softmax(logits, dim=1)

print(f"\nLogits: {logits}")
print(f"Probs: {probs}")
print(f"Prediction: {'ANOMALY' if logits.argmax(dim=1).item() == 1 else 'NORMAL'}")

In [19]:
# Build evidence store from training sessions
evidence_store = EvidenceStore(dataset="HDFS")
evidence_store.build_from_sessions(train_sessions, show_progress=True)

# Show statistics
stats = evidence_store.stats()
print("\nEvidence Store Stats:")
for key, value in stats.items():
    print(f"  {key}: {value:,}" if isinstance(value, int) else f"  {key}: {value}")

Building evidence store: 100%|██████████| 402542/402542 [02:39<00:00, 2521.55it/s]


Evidence store built with 402542 documents

Evidence Store Stats:
  total_documents: 402,542
  normal_documents: 390,756
  anomaly_documents: 11,786
  by_evidence_type: {'session': 402542, 'signature': 0, 'profile': 0}
  avg_text_length: 2066.1526250676948
  min_text_length: 208
  max_text_length: 29,077


## 5. Retriever (RAG)

In [ ]:
# Build BM25 retriever
retriever = BM25Retriever(evidence_store)
retriever.build_index()

In [ ]:
# Find a test session to use for demo
if predicted_anomalies:
    # Use first predicted anomaly
    test_session = sample_sessions[screener_outputs.index(predicted_anomalies[0])]
    scr_output = predicted_anomalies[0]
else:
    # Fall back to a ground truth anomaly
    gt_anomaly_sessions = [(s, o) for s, o in zip(sample_sessions, screener_outputs) if s.label == 1]
    if gt_anomaly_sessions:
        test_session, scr_output = gt_anomaly_sessions[0]
        print(f"Using ground truth anomaly: {test_session.session_id}")
    else:
        test_session = sample_sessions[0]
        scr_output = screener_outputs[0]
        print(f"Using first session: {test_session.session_id}")

# Test mixed retrieval
print(f"\nTest session: {test_session.session_id}")
print(f"Session lines: {len(test_session.lines)}")
print(f"Ground truth: {'ANOMALY' if test_session.label == 1 else 'NORMAL'}")

# Mixed retrieval (4 anomaly + 1 normal)
print("\n=== Mixed Retrieval (4 anomaly + 1 normal) ===")
mixed_hits = retriever.retrieve_for_session_mixed(test_session, top_k_anomaly=4, top_k_normal=1)
for h in mixed_hits:
    label = "anomaly" if h.metadata.get("label") == 1 else "normal"
    print(f"  {h.evidence_id[:25]:25s} label={label:7s} score={h.score:.2f}")

## 6. Prompt Builder

In [ ]:
# Initialize prompt builder
builder = PromptBuilder(
    max_log_lines=20,
    max_chars_per_evidence=500,
    max_evidence_items=5
)

# Build prompt
system_prompt, user_prompt = builder.build_prompt(
    session=test_session,
    screener_output=scr_output,
    evidence_hits=mixed_hits
)

print("=== SYSTEM PROMPT ===")
print(system_prompt[:500])
print("...")

print("\n=== USER PROMPT (truncated) ===")
print(user_prompt[:1500])
print("...")

## 7. LLM Client

In [ ]:
# Initialize LLM client
llm_client = LLMClient(
    provider="ollama",
    model="llama3.1:8b",
    temperature=0.1,
    max_tokens=1024,
    timeout=120
)

# Check availability
if llm_client.is_available():
    print(f"LLM ({llm_client.model}) is available")
else:
    print(f"LLM not available. Start with: ollama serve")

In [ ]:
# Generate explanation
print("Generating explanation...")
start = time.time()

response = llm_client.generate(
    prompt=user_prompt,
    system_prompt=system_prompt,
    json_mode=True
)

elapsed = time.time() - start
print(f"Done in {elapsed:.2f}s")
print(f"Tokens: {response.total_tokens}")

In [ ]:
# Parse and display the explanation
explanation_dict = json.loads(response.content)

print("=" * 60)
print("LLM EXPLANATION")
print("=" * 60)

# Signature
if 'signature' in explanation_dict:
    sig = explanation_dict['signature']
    print(f"\nSignature: {sig.get('name', 'N/A')}")

print(f"\nPrediction: {explanation_dict.get('prediction')}")
print(f"Summary: {explanation_dict.get('summary')}")
print(f"\nClaims ({len(explanation_dict.get('claims', []))}):")    

for i, claim in enumerate(explanation_dict.get('claims', []), 1):
    print(f"\n  [{i}] {claim.get('type', 'observation')}")
    print(f"      {claim.get('claim', 'N/A')}")
    print(f"      Evidence: {claim.get('evidence_ids', [])} | Spans: {claim.get('evidence_spans', [])}")

## 8. Verifier

In [ ]:
# Initialize verifier
verifier = Verifier()

# Convert dict to TraceExplanation
sig_dict = explanation_dict.get('signature')
signature = None
if sig_dict:
    signature = Signature(
        name=sig_dict.get('name', 'UNKNOWN'),
        matched_evidence_ids=sig_dict.get('matched_evidence_ids', [])
    )

trace_exp = TraceExplanation(
    prediction=explanation_dict.get('prediction'),
    summary=explanation_dict.get('summary'),
    signature=signature,
    claims=[Claim(
        type=c.get('type', 'observation'),
        claim=c.get('claim'),
        evidence_ids=c.get('evidence_ids', []),
        evidence_spans=c.get('evidence_spans', [])
    ) for c in explanation_dict.get('claims', [])],
    insufficient_evidence=explanation_dict.get('insufficient_evidence', False)
)

# Build evidence ID mapping and verify
evidence_id_mapping = builder.build_evidence_id_mapping(test_session, mixed_hits)
query_session_text = "\n".join(test_session.lines)  # E0 text

verification = verifier.verify(
    explanation=trace_exp,
    evidence_hits=mixed_hits,
    evidence_id_mapping=evidence_id_mapping,
    query_session_text=query_session_text
)

print("=" * 60)
print("VERIFICATION RESULT")
print("=" * 60)
print(f"\nPassed: {verification.passed}")
print(f"Checks: {verification.passed_checks}/{verification.total_checks} passed")

## 9. Complete Pipeline Function

In [ ]:
def explain_session(
    session: Session,
    screener_output: ScreenerOutput,
    retriever: BM25Retriever,
    builder: PromptBuilder,
    llm_client: LLMClient,
    verifier: Verifier
) -> dict:
    """
    Generate and verify an explanation for an anomalous session.
    """
    start = time.time()
    
    # 1. Retrieve evidence (mixed: 4 anomaly + 1 normal)
    evidence_hits = retriever.retrieve_for_session_mixed(
        session, 
        top_k_anomaly=4, 
        top_k_normal=1
    )
    
    # 2. Build prompt
    system_prompt, user_prompt = builder.build_prompt(
        session=session,
        screener_output=screener_output,
        evidence_hits=evidence_hits
    )
    
    # 3. Call LLM
    response = llm_client.generate(
        prompt=user_prompt,
        system_prompt=system_prompt,
        json_mode=True
    )
    
    # 4. Parse response
    try:
        explanation_dict = json.loads(response.content)
        parse_success = True
    except json.JSONDecodeError:
        explanation_dict = {"prediction": "anomaly", "summary": "Parse error", "claims": [], "signature": None}
        parse_success = False
    
    # 5. Convert to TraceExplanation
    sig_dict = explanation_dict.get('signature')
    signature = Signature(
        name=sig_dict.get('name', 'UNKNOWN'),
        matched_evidence_ids=sig_dict.get('matched_evidence_ids', [])
    ) if sig_dict else None
    
    trace_exp = TraceExplanation(
        prediction=explanation_dict.get('prediction'),
        summary=explanation_dict.get('summary'),
        signature=signature,
        claims=[Claim(
            type=c.get('type', 'observation'),
            claim=c.get('claim'),
            evidence_ids=c.get('evidence_ids', []),
            evidence_spans=c.get('evidence_spans', [])
        ) for c in explanation_dict.get('claims', [])],
        insufficient_evidence=explanation_dict.get('insufficient_evidence', False)
    )
    
    # 6. Verify (pass E0 query session text for keyword matching)
    evidence_id_mapping = builder.build_evidence_id_mapping(session, evidence_hits)
    query_session_text = "\n".join(session.lines)
    verification = verifier.verify(
        trace_exp, evidence_hits, evidence_id_mapping,
        query_session_text=query_session_text
    )
    
    elapsed = time.time() - start
    
    return {
        'session_id': session.session_id,
        'explanation': explanation_dict,
        'signature': signature.name if signature else None,
        'verification_passed': verification.passed,
        'verification_details': {
            'total_checks': verification.total_checks,
            'passed_checks': verification.passed_checks,
            'failed_checks': verification.failed_checks,
            'warning_checks': verification.warning_checks
        },
        'parse_success': parse_success,
        'tokens': response.total_tokens,
        'latency_ms': elapsed * 1000
    }

print("Pipeline function defined: explain_session()")

In [ ]:
# Test the complete pipeline function
result = explain_session(
    session=test_session,
    screener_output=scr_output,
    retriever=retriever,
    builder=builder,
    llm_client=llm_client,
    verifier=verifier
)

print("=" * 60)
print("PIPELINE RESULT")
print("=" * 60)
print(f"Session: {result['session_id']}")
print(f"Parse success: {result['parse_success']}")
print(f"Verification: {'PASSED' if result['verification_passed'] else 'FAILED'}")
print(f"Tokens: {result['tokens']}")
print(f"Latency: {result['latency_ms']:.0f}ms")

## 10. Batch Processing

In [ ]:
# Batch process predicted anomalies
from tqdm import tqdm
from collections import Counter

# Get all predicted anomalies (or use ground truth if screener predicts none)
if predicted_anomalies:
    anomaly_pairs = [
        (sample_sessions[i], screener_outputs[i])
        for i, o in enumerate(screener_outputs)
        if o.is_anomaly
    ]
else:
    # Fall back to ground truth anomalies for demo
    anomaly_pairs = [
        (sample_sessions[i], screener_outputs[i])
        for i, s in enumerate(sample_sessions)
        if s.label == 1
    ][:10]  # Limit to 10 for demo
    print(f"Using {len(anomaly_pairs)} ground truth anomalies for demo")

print(f"Processing {len(anomaly_pairs)} anomalies...")

batch_results = []
for session, scr_output in tqdm(anomaly_pairs, desc="Explaining"):
    result = explain_session(
        session=session,
        screener_output=scr_output,
        retriever=retriever,
        builder=builder,
        llm_client=llm_client,
        verifier=verifier
    )
    batch_results.append(result)

# Summary statistics
passed = sum(1 for r in batch_results if r['verification_passed'])
total_tokens = sum(r['tokens'] for r in batch_results)
avg_latency = sum(r['latency_ms'] for r in batch_results) / len(batch_results) if batch_results else 0

print("\n" + "=" * 60)
print("BATCH RESULTS")
print("=" * 60)
print(f"\nSessions processed: {len(batch_results)}")
print(f"Verification passed: {passed} / {len(batch_results)} ({passed/len(batch_results):.1%})")
print(f"Total tokens: {total_tokens:,}")
print(f"Avg tokens/session: {total_tokens/len(batch_results):.0f}")
print(f"Avg latency: {avg_latency:.0f}ms")

# Signature distribution
signatures = [r['signature'] for r in batch_results if r['signature']]
sig_counts = Counter(signatures)
print(f"\nSignature distribution:")
for sig, count in sig_counts.most_common(10):
    print(f"  {sig}: {count}")

# Verification breakdown
total_checks = sum(r['verification_details']['total_checks'] for r in batch_results)
passed_checks = sum(r['verification_details']['passed_checks'] for r in batch_results)
print(f"\nVerification checks: {passed_checks}/{total_checks} ({passed_checks/total_checks:.1%})")

## 11. Summary

# HDFS Explanation Pipeline

Complete Screener-Reasoner pipeline for HDFS dataset with:
- Evidence-grounded explanations
- Signature extraction
- Line-level evidence spans

Based on `03_pipeline_complete.ipynb` (BGL version).

## 1. Imports

In [8]:
# Standard library
import sys
import json
import time
from pathlib import Path
from importlib import reload

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

# Project imports
from src.data_loader import HDFSDataLoader, Session
from src import screener as screener_module
reload(screener_module)  # Force reload after config fix
from src.screener import Screener, ScreenerOutput
from src.evidence_store import EvidenceStore
from src.retriever import BM25Retriever, Retriever
from src.signature_generator import SignatureGenerator
from src.prompt_builder import PromptBuilder, TraceExplanation, Claim, Signature, format_evidence_block
from src.llm_client import LLMClient
from src.verifier import Verifier

print("✓ All imports successful")

✓ All imports successful


## 2. Data Loading

In [5]:
# Load HDFS dataset
loader = HDFSDataLoader(
    log_file='../logs/HDFS.log',
    label_file='../logs/anomaly_label_HDFS.csv'
)
loader.load()

# Get splits
train_sessions = loader.get_train()
test_sessions = loader.get_test()

# Statistics
train_anomaly = sum(1 for s in train_sessions if s.label == 1)
test_anomaly = sum(1 for s in test_sessions if s.label == 1)

print(f"Train: {len(train_sessions):,} sessions ({train_anomaly:,} anomalies, {train_anomaly/len(train_sessions):.2%})")
print(f"Test:  {len(test_sessions):,} sessions ({test_anomaly:,} anomalies, {test_anomaly/len(test_sessions):.2%})")

Loading HDFS logs from: ../logs/HDFS.log
Loading labels from: ../logs/anomaly_label_HDFS.csv


Reading HDFS logs: 11175629it [00:10, 1097940.61it/s]


Found 575061 unique blocks
Train: 402,542 sessions (11,786 anomalies, 2.93%)
Test:  86,260 sessions (2,526 anomalies, 2.93%)


## 3. Screener

In [9]:
# Load pre-trained screener model
screener = Screener.from_pretrained(
    dataset="HDFS",
    model_path="../best_model_HDFS/best_model_HDFS20250804_201746.pth"
)

total_params = sum(p.numel() for p in screener.model.parameters())
print(f"Model parameters: {total_params:,}")

Loading Screener for HDFS on cuda
Loading cl100k_base (GPT-4) tokenizer...
Loading model weights from: ../best_model_HDFS/best_model_HDFS20250804_201746.pth
Model loaded! Parameters: 15,501,506
Model parameters: 15,501,506


In [17]:
# NOTE: The HDFS screener model appears to have accuracy issues (predicting all normal)
# For testing the explanation pipeline, we'll use ground-truth anomalies directly
# This is a workaround to test the explanation generation pipeline

# Find ground-truth anomalies in test set
test_anomalies = [s for s in test_sessions if s.label == 1]
print(f"Ground-truth anomalies in test set: {len(test_anomalies):,}")

# Take a sample for testing
sample_size = 20
sample_sessions = test_anomalies[:sample_size]
print(f"\\nUsing {sample_size} ground-truth anomalies for explanation pipeline testing")

# Create mock screener outputs (since actual screener has issues)
predicted_anomalies = []
for s in sample_sessions:
    # Create a mock ScreenerOutput marking it as anomaly
    mock_output = ScreenerOutput(
        session_id=s.session_id,
        pred=1,
        logits=[0.0, 1.0],
        prob=[0.0, 1.0],  # Note: prob not probs
        margin=1.0
    )
    predicted_anomalies.append(mock_output)

print(f"Mock screener outputs created: {len(predicted_anomalies)}")

# Also keep screener_outputs reference for downstream compatibility
screener_outputs = predicted_anomalies

Ground-truth anomalies in test set: 2,526
\nUsing 20 ground-truth anomalies for explanation pipeline testing
Mock screener outputs created: 20


In [15]:
# Debug: Manual inference
import torch

test_idx = 4  # First true anomaly  
test_sess = sample_sessions[test_idx]

# Tokenize
input_ids, segment_ids = screener._tokenize_session(test_sess.lines)

# Convert to tensors
input_ids_tensor = torch.tensor([input_ids], dtype=torch.long).to(screener.device)
segment_ids_tensor = torch.tensor([segment_ids], dtype=torch.long).to(screener.device)

# Pad to max length
if input_ids_tensor.size(1) < screener.max_seq_len:
    pad_size = screener.max_seq_len - input_ids_tensor.size(1)
    input_ids_tensor = torch.nn.functional.pad(input_ids_tensor, (0, pad_size), value=0)
    segment_ids_tensor = torch.nn.functional.pad(segment_ids_tensor, (0, pad_size), value=0)

print(f"Input shape: {input_ids_tensor.shape}")
print(f"max_seq_len: {screener.max_seq_len}")

attention_mask = (input_ids_tensor != 0).long()

# Inference
with torch.no_grad():
    logits = screener.model(input_ids_tensor, segment_ids_tensor, attention_mask)
    probs = torch.softmax(logits, dim=1)
    
print(f"\\nLogits: {logits}")
print(f"Probabilities: {probs}")
print(f"Prediction: {'ANOMALY' if logits.argmax(dim=1).item() == 1 else 'NORMAL'}")

Input shape: torch.Size([1, 15166])
max_seq_len: 15166
\nLogits: tensor([[ 8.7946, -8.1846]], device='cuda:0')
Probabilities: tensor([[1.0000e+00, 4.2267e-08]], device='cuda:0')
Prediction: NORMAL


## 4. Evidence Store

In [ ]:
# Build evidence store from training sessions
evidence_store = EvidenceStore(dataset="HDFS")
evidence_store.build_from_sessions(train_sessions, show_progress=True)

# Show statistics
stats = evidence_store.stats()
print("\nEvidence Store Stats:")
for key, value in stats.items():
    print(f"  {key}: {value:,}" if isinstance(value, int) else f"  {key}: {value}")

## 5. Retriever (RAG)

In [ ]:
# Build BM25 retriever
retriever = BM25Retriever(evidence_store)
retriever.build_index()

In [ ]:
# Test mixed retrieval on an anomalous session
test_session = sample_sessions[screener_outputs.index(predicted_anomalies[0])]

# Standard retrieval (top-5 any label)
print("=== Standard Retrieval (top-5 any) ===")
standard_hits = retriever.retrieve_for_session(test_session, top_k=5)
for h in standard_hits:
    label = "anomaly" if h.metadata.get("label") == 1 else "normal"
    print(f"  {h.evidence_id[:25]:25s} label={label:7s} score={h.score:.2f}")

# Mixed retrieval (4 anomaly + 1 normal)
print("\n=== Mixed Retrieval (4 anomaly + 1 normal) ===")
mixed_hits = retriever.retrieve_for_session_mixed(test_session, top_k_anomaly=4, top_k_normal=1)
for h in mixed_hits:
    label = "anomaly" if h.metadata.get("label") == 1 else "normal"
    print(f"  {h.evidence_id[:25]:25s} label={label:7s} score={h.score:.2f}")

## 6. Prompt Builder

In [ ]:
# Initialize prompt builder
builder = PromptBuilder(
    max_log_lines=20,
    max_chars_per_evidence=500,
    max_evidence_items=5
)

# Create a mock screener output for the test session
scr_output = predicted_anomalies[0]

# Build prompt
system_prompt, user_prompt = builder.build_prompt(
    session=test_session,
    screener_output=scr_output,
    evidence_hits=mixed_hits
)

print("=== SYSTEM PROMPT ===")
print(system_prompt[:500])
print("...")

print("\n=== USER PROMPT (truncated) ===")
print(user_prompt[:1500])
print("...")

## 7. LLM Client

In [ ]:
# Initialize LLM client
llm_client = LLMClient(
    provider="ollama",
    model="llama3.1:8b",
    temperature=0.1,
    max_tokens=1024,
    timeout=120
)

# Check availability
if llm_client.is_available():
    print(f"✓ LLM ({llm_client.model}) is available")
else:
    print(f"✗ LLM not available. Start with: ollama serve")

In [ ]:
# Generate explanation
print("Generating explanation...")
start = time.time()

response = llm_client.generate(
    prompt=user_prompt,
    system_prompt=system_prompt,
    json_mode=True
)

elapsed = time.time() - start
print(f"Done in {elapsed:.2f}s")
print(f"Tokens: {response.total_tokens}")

In [ ]:
# Parse and display the explanation
explanation_dict = json.loads(response.content)

print("=" * 60)
print("LLM EXPLANATION")
print("=" * 60)

# Signature
if 'signature' in explanation_dict:
    sig = explanation_dict['signature']
    print(f"\nSignature: {sig.get('name', 'N/A')}")

print(f"\nPrediction: {explanation_dict.get('prediction')}")
print(f"Summary: {explanation_dict.get('summary')}")
print(f"\nClaims ({len(explanation_dict.get('claims', []))}):")    

for i, claim in enumerate(explanation_dict.get('claims', []), 1):
    print(f"\n  [{i}] {claim.get('type', 'observation')}")
    print(f"      {claim.get('claim', 'N/A')}")
    print(f"      Evidence: {claim.get('evidence_ids', [])} | Spans: {claim.get('evidence_spans', [])}")

## 8. Verifier

In [ ]:
# Initialize verifier
verifier = Verifier()

# Convert dict to TraceExplanation
sig_dict = explanation_dict.get('signature')
signature = None
if sig_dict:
    signature = Signature(
        name=sig_dict.get('name', 'UNKNOWN'),
        matched_evidence_ids=sig_dict.get('matched_evidence_ids', [])
    )

trace_exp = TraceExplanation(
    prediction=explanation_dict.get('prediction'),
    summary=explanation_dict.get('summary'),
    signature=signature,
    claims=[Claim(
        type=c.get('type', 'observation'),
        claim=c.get('claim'),
        evidence_ids=c.get('evidence_ids', []),
        evidence_spans=c.get('evidence_spans', [])
    ) for c in explanation_dict.get('claims', [])],
    insufficient_evidence=explanation_dict.get('insufficient_evidence', False)
)

# Build evidence ID mapping and verify
evidence_id_mapping = builder.build_evidence_id_mapping(test_session, mixed_hits)
query_session_text = "\n".join(test_session.lines)  # E0 text

verification = verifier.verify(
    explanation=trace_exp,
    evidence_hits=mixed_hits,
    evidence_id_mapping=evidence_id_mapping,
    query_session_text=query_session_text
)

print("=" * 60)
print("VERIFICATION RESULT")
print("=" * 60)
print(f"\nPassed: {verification.passed}")
print(f"Checks: {verification.passed_checks}/{verification.total_checks} passed")

## 9. Complete Pipeline Function

In [ ]:
def explain_session(
    session: Session,
    screener_output: ScreenerOutput,
    retriever: BM25Retriever,
    builder: PromptBuilder,
    llm_client: LLMClient,
    verifier: Verifier
) -> dict:
    """
    Generate and verify an explanation for an anomalous session.
    
    Pipeline steps:
    1. Retrieve evidence (4 anomaly + 1 normal) via BM25
    2. Build structured prompt with evidence
    3. Call LLM to generate explanation
    4. Parse and validate response
    5. Verify faithfulness against evidence
    
    Returns:
        dict with explanation, verification status, and metrics
    """
    start = time.time()
    
    # 1. Retrieve evidence (mixed: 4 anomaly + 1 normal)
    evidence_hits = retriever.retrieve_for_session_mixed(
        session, 
        top_k_anomaly=4, 
        top_k_normal=1
    )
    
    # 2. Build prompt
    system_prompt, user_prompt = builder.build_prompt(
        session=session,
        screener_output=screener_output,
        evidence_hits=evidence_hits
    )
    
    # 3. Call LLM
    response = llm_client.generate(
        prompt=user_prompt,
        system_prompt=system_prompt,
        json_mode=True
    )
    
    # 4. Parse response
    try:
        explanation_dict = json.loads(response.content)
        parse_success = True
    except json.JSONDecodeError:
        explanation_dict = {"prediction": "anomaly", "summary": "Parse error", "claims": [], "signature": None}
        parse_success = False
    
    # 5. Convert to TraceExplanation
    sig_dict = explanation_dict.get('signature')
    signature = Signature(
        name=sig_dict.get('name', 'UNKNOWN'),
        matched_evidence_ids=sig_dict.get('matched_evidence_ids', [])
    ) if sig_dict else None
    
    trace_exp = TraceExplanation(
        prediction=explanation_dict.get('prediction'),
        summary=explanation_dict.get('summary'),
        signature=signature,
        claims=[Claim(
            type=c.get('type', 'observation'),
            claim=c.get('claim'),
            evidence_ids=c.get('evidence_ids', []),
            evidence_spans=c.get('evidence_spans', [])
        ) for c in explanation_dict.get('claims', [])],
        insufficient_evidence=explanation_dict.get('insufficient_evidence', False)
    )
    
    # 6. Verify (pass E0 query session text for keyword matching)
    evidence_id_mapping = builder.build_evidence_id_mapping(session, evidence_hits)
    query_session_text = "\n".join(session.lines)
    verification = verifier.verify(
        trace_exp, evidence_hits, evidence_id_mapping,
        query_session_text=query_session_text
    )
    
    elapsed = time.time() - start
    
    return {
        'session_id': session.session_id,
        'explanation': explanation_dict,
        'signature': signature.name if signature else None,
        'verification_passed': verification.passed,
        'verification_details': {
            'total_checks': verification.total_checks,
            'passed_checks': verification.passed_checks,
            'failed_checks': verification.failed_checks,
            'warning_checks': verification.warning_checks
        },
        'parse_success': parse_success,
        'tokens': response.total_tokens,
        'latency_ms': elapsed * 1000
    }

print("Pipeline function defined: explain_session()")

In [ ]:
# Test the complete pipeline function
result = explain_session(
    session=test_session,
    screener_output=scr_output,
    retriever=retriever,
    builder=builder,
    llm_client=llm_client,
    verifier=verifier
)

print("=" * 60)
print("PIPELINE RESULT")
print("=" * 60)
print(f"Session: {result['session_id']}")
print(f"Parse success: {result['parse_success']}")
print(f"Verification: {'PASSED ✓' if result['verification_passed'] else 'FAILED ✗'}")
print(f"Tokens: {result['tokens']}")
print(f"Latency: {result['latency_ms']:.0f}ms")

## 10. Batch Processing

In [ ]:
# Batch process predicted anomalies
from tqdm import tqdm
from collections import Counter

# Get all predicted anomalies from screened sessions
anomaly_pairs = [
    (sample_sessions[i], screener_outputs[i])
    for i, o in enumerate(screener_outputs)
    if o.is_anomaly
]

print(f"Processing {len(anomaly_pairs)} predicted anomalies...")

batch_results = []
for session, scr_output in tqdm(anomaly_pairs, desc="Explaining"):
    result = explain_session(
        session=session,
        screener_output=scr_output,
        retriever=retriever,
        builder=builder,
        llm_client=llm_client,
        verifier=verifier
    )
    batch_results.append(result)

# Summary statistics
passed = sum(1 for r in batch_results if r['verification_passed'])
total_tokens = sum(r['tokens'] for r in batch_results)
avg_latency = sum(r['latency_ms'] for r in batch_results) / len(batch_results) if batch_results else 0

print("\n" + "=" * 60)
print("BATCH RESULTS")
print("=" * 60)
print(f"\nSessions processed: {len(batch_results)}")
print(f"Verification passed: {passed} / {len(batch_results)} ({passed/len(batch_results):.1%})")
print(f"Total tokens: {total_tokens:,}")
print(f"Avg tokens/session: {total_tokens/len(batch_results):.0f}")
print(f"Avg latency: {avg_latency:.0f}ms")

# Signature distribution
signatures = [r['signature'] for r in batch_results if r['signature']]
sig_counts = Counter(signatures)
print(f"\nSignature distribution:")
for sig, count in sig_counts.most_common(10):
    print(f"  {sig}: {count}")

# Verification breakdown
total_checks = sum(r['verification_details']['total_checks'] for r in batch_results)
passed_checks = sum(r['verification_details']['passed_checks'] for r in batch_results)
print(f"\nVerification checks: {passed_checks}/{total_checks} ({passed_checks/total_checks:.1%})")

## 11. Summary

# HDFS Explanation Pipeline

Complete Screener-Reasoner pipeline for HDFS dataset with:
- Evidence-grounded explanations
- Signature extraction
- Line-level evidence spans

Based on `03_pipeline_complete.ipynb` (BGL version).

In [2]:
# Standard library
import sys
import json
import time
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

# Project imports
from src.data_loader import HDFSDataLoader, Session
from src.screener import Screener, ScreenerOutput
from src.evidence_store import EvidenceStore
from src.retriever import BM25Retriever, Retriever
from src.signature_generator import SignatureGenerator
from src.prompt_builder import PromptBuilder, TraceExplanation, Claim, format_evidence_block
from src.llm_client import LLMClient
from src.verifier import Verifier

print("✓ All imports successful")

ImportError: cannot import name 'ExplanationVerifier' from 'src.verifier' (/home/dave/agentic-log-explanations/notebooks/../src/verifier.py)

## 1. Load HDFS Data

In [ ]:
# Load HDFS dataset
loader = HDFSDataLoader(
    log_file='../logs/HDFS.log',
    label_file='../logs/anomaly_label_HDFS.csv'
)
train_sessions, val_sessions, test_sessions = loader.load_and_split()

print(f"Train: {len(train_sessions):,} sessions")
print(f"Val: {len(val_sessions):,} sessions")
print(f"Test: {len(test_sessions):,} sessions")

In [ ]:
# Check anomaly ratio
train_anomalies = sum(1 for s in train_sessions if s.label == 1)
test_anomalies = sum(1 for s in test_sessions if s.label == 1)

print(f"Train anomalies: {train_anomalies:,} ({train_anomalies/len(train_sessions):.2%})")
print(f"Test anomalies: {test_anomalies:,} ({test_anomalies/len(test_sessions):.2%})")

## 2. Load Screener Model

In [ ]:
# Load pre-trained screener model
screener = Screener.from_pretrained(
    dataset="HDFS",
    model_path="../best_model_HDFS/best_model_HDFS20250804_201746.pth"
)

total_params = sum(p.numel() for p in screener.model.parameters())
print(f"Model parameters: {total_params:,}")

## 3. Build Evidence Store

In [ ]:
# Build evidence store from training sessions
evidence_store = EvidenceStore(dataset="HDFS")
evidence_store.build_from_sessions(train_sessions, show_progress=True)

# Show statistics
stats = evidence_store.stats()
print("\nEvidence Store Stats:")
for key, value in stats.items():
    print(f"  {key}: {value:,}" if isinstance(value, int) else f"  {key}: {value}")

## 4. Build Retriever

In [ ]:
# Build BM25 retriever
retriever = BM25Retriever(evidence_store)
print(f"BM25 index built with {len(evidence_store.documents):,} documents")

## 5. Initialize LLM & Verifier

In [ ]:
# Initialize components
llm = OllamaClient(model='llama3.1:8b')
prompt_builder = PromptBuilder()
verifier = ExplanationVerifier()

# Test LLM connection
if llm.is_available():
    print("✓ LLM is available")
else:
    print("✗ LLM not available")

## 6. Screen Test Sessions

In [ ]:
# Screen a subset of test sessions
MAX_SESSIONS = 50  # Adjust as needed
test_subset = test_sessions[:MAX_SESSIONS]

print(f"Screening {len(test_subset)} sessions...")
results = screener.screen_batch(test_subset)

# Find predicted anomalies
predicted_anomalies = [
    (session, result) 
    for session, result in zip(test_subset, results) 
    if result.pred == 1
]

print(f"\nPredicted anomalies: {len(predicted_anomalies)} / {len(test_subset)} ({len(predicted_anomalies)/len(test_subset):.1%})")

## 7. Explain Anomalies

In [ ]:
def explain_session(session, screener_result, retriever, llm, prompt_builder,
                    top_k_anomaly=4, top_k_normal=1):
    """
    Generate explanation for a single session.
    
    Returns:
        dict with keys: session_id, explanation, evidence, latency_ms, tokens
    """
    # Retrieve evidence (mixed: anomaly + normal)
    evidence = retriever.retrieve_for_session_mixed(
        session,
        top_k_anomaly=top_k_anomaly,
        top_k_normal=top_k_normal,
        exclude_self=True
    )
    
    # Build prompt
    system_prompt, user_prompt = prompt_builder.build_prompt(
        session=session,
        evidence=evidence,
        screener_output=screener_result
    )
    
    # Call LLM
    start = time.time()
    response = llm.generate(system_prompt, user_prompt)
    latency_ms = (time.time() - start) * 1000
    
    # Parse response
    explanation = prompt_builder.parse_response(response.text)
    
    return {
        'session_id': session.session_id,
        'explanation': explanation,
        'evidence': evidence,
        'latency_ms': latency_ms,
        'tokens': response.tokens_used
    }

In [ ]:
# Generate explanations for all predicted anomalies
explanations = []

for session, result in tqdm(predicted_anomalies, desc="Explaining"):
    try:
        exp = explain_session(session, result, retriever, llm, prompt_builder)
        explanations.append(exp)
    except Exception as e:
        print(f"Failed {session.session_id}: {e}")

print(f"\nGenerated {len(explanations)} explanations")

## 8. View Sample Explanation

In [ ]:
# Display first explanation
if explanations:
    exp = explanations[0]
    print(f"Session: {exp['session_id']}")
    print(f"Latency: {exp['latency_ms']:.0f}ms")
    print(f"Tokens: {exp['tokens']}")
    print("\n" + "="*60)
    
    if exp['explanation']:
        expl = exp['explanation']
        print(f"Prediction: {expl.prediction}")
        print(f"Summary: {expl.summary}")
        
        if expl.signature:
            print(f"\nSignature: {expl.signature.name}")
            print(f"  Matched evidence: {expl.signature.matched_evidence_ids}")
        
        print(f"\nClaims ({len(expl.claims)}):")
        for i, claim in enumerate(expl.claims, 1):
            print(f"  [{i}] {claim.claim}")
            print(f"      Type: {claim.type}")
            print(f"      Evidence: {claim.evidence_ids}")
            if claim.evidence_spans:
                print(f"      Spans: {claim.evidence_spans}")
    else:
        print("Parse failed")

## 9. Verify Explanations

In [ ]:
# Verify all explanations
from src.normalizer import LogNormalizer
normalizer = LogNormalizer()

total_checks = 0
passed_checks = 0
warnings = 0

for exp in explanations:
    if not exp['explanation']:
        continue
    
    # Get query session text for E0 verification
    session = next(s for s, _ in predicted_anomalies if s.session_id == exp['session_id'])
    query_text = normalizer.normalize(session.lines)
    
    # Verify
    result = verifier.verify(
        explanation=exp['explanation'],
        evidence=exp['evidence'],
        query_session_text=query_text
    )
    
    total_checks += result.total_checks
    passed_checks += result.passed_checks
    warnings += len(result.warnings)

print(f"Verification Results:")
print(f"  Total checks: {total_checks}")
print(f"  Passed: {passed_checks} ({passed_checks/total_checks:.1%})")
print(f"  Failed: {total_checks - passed_checks}")
print(f"  Warnings: {warnings}")

## 10. Signature Analysis

In [ ]:
# Analyze signature distribution
from collections import Counter

signatures = []
for exp in explanations:
    if exp['explanation'] and exp['explanation'].signature:
        signatures.append(exp['explanation'].signature.name)

sig_counts = Counter(signatures)

print(f"Signature Distribution ({len(sig_counts)} unique):")
for sig, count in sig_counts.most_common(10):
    print(f"  {sig}: {count}")

## 11. Summary Statistics

In [ ]:
import numpy as np

latencies = [e['latency_ms'] for e in explanations]
tokens = [e['tokens'] for e in explanations]

print("="*60)
print("HDFS PIPELINE SUMMARY")
print("="*60)

print(f"\nSessions:")
print(f"  Screened: {len(test_subset)}")
print(f"  Anomalies: {len(predicted_anomalies)} ({len(predicted_anomalies)/len(test_subset):.1%})")

print(f"\nExplanations:")
print(f"  Generated: {len(explanations)}")
print(f"  With signature: {len(signatures)}")

print(f"\nVerification:")
print(f"  Pass rate: {passed_checks/total_checks:.1%}")
print(f"  Warnings: {warnings}")

print(f"\nLatency:")
print(f"  Avg: {np.mean(latencies):.0f}ms")
print(f"  P95: {np.percentile(latencies, 95):.0f}ms")

print(f"\nTokens:")
print(f"  Total: {sum(tokens):,}")
print(f"  Avg: {np.mean(tokens):.0f}")